# Publish contend based on one model

This notebook contains the steps to generate the SSOT from an ODM model and render content.

This notebook will be executed by the run.bat

In [ ]:
odm_source_folder = 'IM'
destination_folder = 'content'

# Skip generation of the SSOT from ODM? 
skip_ssot_generation = False

# Location of the python tools
tools_path = 'pythonWork/pythonSource'

In [ ]:
import sys
import os
from pathlib import Path
import glob
import json
from contextlib import closing

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/generator-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

In [ ]:
sys.argv[0]

In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Generate SSOT and diagrams from ODM model')
parser.add_argument('--model', '-m', dest='source_folder', default=odm_source_folder)
parser.add_argument('--destination', dest='destination_folder', default=destination_folder)
parser.add_argument('--skip-odm', '-n', action='store_true', dest='skip_odm',
                    help="Skip generation/update of single point of documentation (SPOD) from ODM")
parser.add_argument('--skip-web', '-w', action='store_true', dest='skip_web',
                    help="Skip generation of static web content")
parser.add_argument('--confluence', '-c', action='store_true', dest='confluence',
                    help="Skip generation of Confluence content")
parser.add_argument('--sharepoint', '-s', action='store_true', dest='sharepoint',
                    help="Enable generation of Sharepoint content")
parser.add_argument('--sparx-ea', '-e', action='store_true', dest='sparx_ea',
                    help="Enable generation Sparx Enterprise Architect compatible UML/XMI export")
parser.add_argument('--dont-merge', action="store_true", dest="clean_slate",
                    help="Do not merge! Existing database is renamed to .bkp")
parser.add_argument('--all', action="store_true", dest="all", help="Generate all possible outputs")
parser.add_argument('--spod-only', action="store_true", dest="spod_only", help="Stop after SPOD creation")
parser.add_argument('--tools-path', dest='tools_path', default='pythonWork/pythonSource', help="Tools path")
parser.add_argument('--profile', action='store_true', dest='profile', help="Enable profiling")
parser.add_argument('--languages', '-l', dest='languages', default=None, help="Comma separated list of languages (de,en,fr, ...)\n"
                    "First language will be the 'default'.\n"
                    "If undefinded the languages will be read from SPOD if present or the model.")
parser.add_argument('--link-udpr', dest='udpr_pattern', help="User defined properties matching this pattern key and a hyperlink in the value will override any other links")
parser.add_argument('--version', action='store_true')
parser.add_argument('--verbose', action='store_true', help="Log all information to console and logfile")

arguments = argparse.Namespace()

if len(sys.argv) > 0 and '.py' in sys.argv[0] and not 'ipykernel' in sys.argv[0]:
    arguments = parser.parse_args()
    odm_source_folder = arguments.source_folder
    destination_folder = arguments.destination_folder
    skip_ssot_generation = arguments.skip_odm
    tools_path = arguments.tools_path
else:
    # in jupyter environment
    assert 'ipykernel' in sys.argv[0], f"Expecting to run in Jupyter environment 🙀"
    odm_source_folder = '../../pythonWork/pythonSource/testenvironment/testmodels/riddle/IM'
    destination_folder = 'content'
    tools_path = os.path.abspath('../../pythonWork/pythonSource')
    skip_ssot_generation = False
    arguments = argparse.Namespace(**{
        'clean_slate': False, 
        'sparx_ea': True, 
        'languages': None, 
        'profile': True,
        'verbose': False,
        'version': False,
        'spod_only': False,
        'udpr_pattern': 'ADONIS'})


In [ ]:
odm_source_folder = '/Users/bue/dev/fyyccim-tools/testdata/fyyccim-refmodels/CRM/IM'
odm_source_folder = '/Users/bue/projects/lawa/ktlu-im/IM'
odm_source_folder = '/Users/bue/projects/geberit/DEAP/IM'
#odm_source_folder = '/Users/bue/projects/sika/Sika-IM/IM'

In [ ]:
if arguments.verbose:
    console_log_handler.setLevel(logging.DEBUG)

In [ ]:
tools_absolute = Path(tools_path).resolve()
logger.debug(f"Working with tools in {tools_absolute}")

# Add toolbox to python library path
sys.path.insert(0, str(tools_absolute))
import SSOT_infra

In [ ]:
if arguments.version:
    ver = SSOT_infra.__version__
    print(f"generator version {ver['TOOLVERSION']} (schema {ver['DBVERSION']})")
    exit(0)

In [ ]:
VERSION_TAG = SSOT_infra.__version__
logger.info(f"Starting generator version {VERSION_TAG}")

logger.debug(f"Configuration: {arguments}")

### Safeguards


In [ ]:
models = glob.glob(odm_source_folder + '/*.[dD][mM][dD]')
if len(models) < 1:
    logger.fatal(f"No model file (*.dmd) found in source folder {os.path.abspath(odm_source_folder)}.")
    exit(2)

model = models[0]
if len(models) > 1:
    for name in models:
        # pick model with shortest name
        if len(name) < len(model):
            model = name
    logger.warning(f"Found {len(models)} models in {os.path.abspath(odm_source_folder)} using {model}.\nModels found: {models}")
    # disable other models?
    for name in models:
        if name != model:
            os.rename(name, os.path.splitext(name)[0] + '.hidden')
    logger.error("Cannot work when several models are in the same IM folder: " + ', '.join(models))
    exit(1)

In [ ]:
base_path = Path(odm_source_folder)
assert os.path.isdir(odm_source_folder), "Cannot find source folder {}".format(odm_source_folder)
project_files = list(base_path.glob('*.dmd'))
assert len(project_files) == 1, "Cannot find exactly 1 ODM .dmd file in source folder {}: {}".format(odm_source_folder,
                                                                                                     project_files)
logger.info(f"Processing the information model in {os.path.abspath(base_path)}")
from IPython.core.display import HTML

HTML(
    '<span style="font-family: Impact; font-size:48px">Processing the information model in<br/><span style="color: darkorange">{0}</span></span>'.format(
        os.path.abspath(odm_source_folder)))

In [ ]:
if not (os.path.exists(tools_path)
        and os.path.isfile(os.path.join(tools_path, 'SSOT_db', 'createDB.py'))):
    logger.fatal(f"Tools not in expected path {tools_path}")
    exit(3)

from SSOT_infra import parameters
global parameters

from IM_WEB.IM_HTML import drawiodiagram
from SSOT_db.IM_JSON import JSModel

In [ ]:
from SSOT_infra import logmessages

def tap_logmessages(message: str):
    logger.warning(message)


logmessages.logtrap = tap_logmessages

In [ ]:
if arguments.languages is not None:
    languages = [x.strip() for x in arguments.languages.split(',')]
else:
    languages = '<from model>'

# Process the datasource

In [ ]:

model_path = Path(model).resolve()
model_name = model_path.stem
logger.info(f"Initializing parameters for model '{model_name}' in {model_path} for languages {languages}") 
parameters.initparam(pbasedirec=str(model_path.parent.resolve()), 
                     pdbfile=str(Path(model_path.parent.parent) / 'DB' / f"{model_name}.db"),
                     pmodelfilepath=str(model_path.parts[-2:]), 
                     pmodelname=model_name)

config_folder_before = None

odm_config_folder = Path(parameters.odmKonfDirec()).resolve()
if not os.path.isdir(odm_config_folder):
    logger.warning(f"Cannot find {odm_config_folder}")
    configuration_path = odm_config_folder.with_name('Configuration')
    parameters.odmKonfDirec(str(configuration_path))
    parameters.odmDefDomainsfilePath(str(Path(configuration_path, parameters.odmdefdomainsfile())))
    logger.warning(f"Patching config folder to {parameters.odmKonfDirec()}")
    if not configuration_path.is_dir():
        logger.warning(f"Missing configuration folder {configuration_path}, patching ...")
        configuration_path.mkdir(exist_ok=True)

assert os.path.isdir(parameters.odmKonfDirec()), f"Configuration folder {parameters.odmKonfDirec()} not found"

In [ ]:
# Ensure base folder exists
base = os.path.split(parameters.dbFilePath())[0]
os.makedirs(base, exist_ok=True)

In [ ]:
if arguments.clean_slate and os.path.isfile(parameters.dbFilePath()):
    existing_database = Path(parameters.dbFilePath())
    logger.info(
        f"Don't merge existing SSOT {existing_database}. It is backed up as {existing_database.with_suffix('.bkp')}")
    existing_database.rename(existing_database.with_suffix('.bkp'))

In [ ]:
from LOAD_MODELS.LOAD_ODM import fillDB
from LOAD_MODELS.LOAD_ODM import transferModel
from SSOT_db.createDB import upgradeDB
from SSOT_db.SQL_INFRA import dbConnect
from SSOT_db.IM_OBJECTS import Language
import atexit
import shutil

def merge():
    fillDB.fillmergedb(pdbfilepath=parameters.dbFilePath(),
                       createnewdb=not Path(parameters.dbFilePath()).is_file(),
                       transferfunction=transferModel.transferODMModel)

def rollback(backup: Path, destination: Path):
    crash = Path(parameters.dbFilePath())
    crash_report = Path(f'log/generator-{run_stamp}.db')
    logger.debug(f"Archiving crash db as {str(crash_report)}")
    crash.replace(crash_report)
    logger.info(f"Restoring {str(destination)} from backup {str(backup)}")    
    backup.replace(destination)
    
    
if not skip_ssot_generation:
    os.makedirs(parameters.dbDirect(), exist_ok=True)

    backup = None
    if os.path.isfile(parameters.dbFilePath()):        
        backup = Path(parameters.dbFilePath()).parent / 'backup.db'
        logger.debug(f"Backing up database to {str(backup)}")
        shutil.copy(parameters.dbFilePath(), backup)
        atexit.register(rollback, backup=backup, destination=parameters.dbFilePath())
        logger.info('Updating database {db} from model {odm}'.format(db=parameters.dbFilePath(), odm=parameters.dbDirect()))
        try:
            applied = upgradeDB()
            if len(applied) > 0:
                logger.info(f"Updated {len(applied)} {applied}")
        except Exception as e:
            print("Cannot upgrade database {db}")
            raise e
        
    
    try:
        languages = []
        if os.path.isfile(parameters.dbFilePath()):
            # load languages from existing SSOT
            with closing(dbConnect.openDBbasic(parameters.dbFilePath())) as dbconn:
                deflang = Language.getdefaultlang().liesdeflangiso2()
                languages = list(set(map(lambda l: str(l.liesdeflangiso2()), Language.select())))
                languages.remove(deflang)
                languages.insert(0, deflang)            
        else:
            # use languages from project comment, if any
            languages, _ = transferModel.read_languages_form_project_comment()
            if languages is None:
                if arguments.languages is None:
                    logger.warning("No languages defined in model and on command line. Using 'en'")
                    languages = ['en']
                else:
                    from collections import OrderedDict
                    cmdlang = list(OrderedDict.fromkeys(map(str.strip, arguments.languages.split(','))))
                    if len(cmdlang) > 1:
                        prime = cmdlang[0]
                        rest = set(cmdlang[1:])
                        try:
                            rest.remove(prime)
                        except ValueError:
                            pass
                        languages = [ prime ] + list(rest)
                    else:
                        languages = cmdlang
        
        logger.info(f"Processing languages {languages}")
        parameters.dbLanguages(','.join(languages))
        parameters.dbDefaultLang(languages[0])
                        
        if arguments.profile:
            logger.info("Starting to profile fillDB.fillmergedb()")
            import cProfile
            profile_stats_file = Path(f'log/generator-{run_stamp}-fillmergedb.prof')
            cProfile.run('merge()', str(profile_stats_file))
            latest_stats_symlink = Path(f'log/generator-lastest-fillmergedb.prof')
            latest_stats_symlink.unlink(missing_ok=True)
            #latest_stats_symlink.hardlink_to(profile_stats_file)
            profile_stats_file.link_to(latest_stats_symlink)
            logger.info(f"Profiling complete. Result stored in {str(profile_stats_file)}. Linked to {str(latest_stats_symlink)}")
        else:
            merge()
           
        with closing(dbConnect.openDBbasic(parameters.dbFilePath())) as dbconn:
            rev = dbConnect.read_git_revision(dbconn)
            assert 'unknown' not in rev, f"Revision should not be unknown '{rev}'"
            logger.info(f"Sucessfully updated {parameters.dbFilePath()} to git revision {rev}")
    except:
        print('Consult logfile {} and {}'.format(logfile, parameters.logfilepath()))
        raise
    
    if backup is not None:
        logger.debug(f"Removing backup on success")
        atexit.unregister(rollback)
        backup.unlink()

In [ ]:
database_file = os.path.abspath(parameters.dbFilePath())
json_file = os.path.splitext(database_file)[0] + '.json'
assert os.path.isfile(json_file), f"SSOT file {json_file} not found"
logger.info(f"Loading SSOT from {json_file}")

## Validation

In [ ]:
with open(json_file) as f:
    data = json.load(f)
assert len(data) > 0, f'Config is empty :-('
assert isinstance(data.get('model'), dict)
assert isinstance(data.get('_imprint_'), dict)
json_rev = data['_imprint_'].get('git-revision')
assert isinstance(data['_imprint_'].get('git-revision'), str)
assert 'unknown' not in data['_imprint_'].get('git-revision')
assert isinstance(data.get('entities'), dict)

## Read database from separate process

In [ ]:
import subprocess

with closing(dbConnect.openDBbasic(parameters.dbFilePath())) as dbconn:
    db_rev = dbConnect.read_git_revision()
    
r = subprocess.check_output(['sqlite3', '-line', str(parameters.dbFilePath()), 'select * from [gitrevision]; select * from [dbversion];'])
logging.debug(f"DB validation: {r.decode()}")

if json_rev == db_rev: 
    logging.info(f"Validated revision '{json_rev}' of json '{str(json_file)}' and db '{str(parameters.dbFilePath())}'")
else:
    logging.error(f"Version discrepancy of json '{str(json_file)}' '{json_rev}' "
                  f"vs. db '{str(parameters.dbFilePath())}' '{db_rev}")

## Stop here if no generation desired

In [ ]:
if arguments.spod_only:
    message = f"\x1b[32mSucessfully\x1b[39m produced documentation in {os.path.abspath(destination_folder)}"
    logging.info(message)
    print(message)
    exit(0)

In [ ]:
# Override languages with the ones present in SPOD json
languages = list(data['languages'].keys())

In [ ]:
jsmodel = JSModel.readfromfile(pfilename=json_file)

In [ ]:
import gettext


class Translator:
    """Translate strings"""
    logger = logging.getLogger("Translator")

    def __init__(self, language: str):
        self.language = language
        self.translator = gettext.translation('confluence-publisher', './locale', fallback=True, languages=[language])
        self.title_format = '{title} - [{language}]'
        self.logger = logging.getLogger("Translator " + language)

    def tr(self, element) -> str:
        if isinstance(element, dict):
            """If the value provided is a field containing translations, use them"""
            text = element.get(self.language)
            if text is None:  #and len(element.values()) > 0:
                result = list(element.values())[0]
                if result is not None:
                    self.logger.warning(f"No translation for {text}. Falling back to {result} from {str(element)}")
                    text = result
            if not text:
                return ''
            # Strip leading translation marker
            #if '*de* ' in text:
            #    text = text.replace('*de* ', '', 1)
            return text

        # Fallback to gettext if not a dict
        if isinstance(element, str):
            translated = self.translator.gettext(element)
            return translated

        self.logger.warning("Cannot translate element '{0}' of type {1}".format(element, type(element)))
        return None

    def gettext(self, text: str):
        result = self.translator.gettext(text)
        if result == text:
            self.logger.warning("No translation for {0}".format(text))
        return result

    def translator(self):
        return self.translator

    def title_language(self, title: str) -> str:
        """Create unique confluence page title per translation"""
        return self.title_format.format(title=title, language=self.language)

    def key_lang(self, key: str) -> str:
        return key + '-' + self.language

    def lang(self) -> str:
        return self.language

In [ ]:
translators = {language: Translator(language) for language in data['languages']}
translators

In [ ]:
def sanitize_filename(name: str) -> str:
    return "".join(c for c in name if c.isalnum() or c in ('.', '-', '_', ' ')).rstrip()

# Render draw.io diagrams

In [ ]:
from lxml import etree

from tqdm.autonotebook import tqdm

content_root = '.'

diagram_count = len(data['diagrams'].keys()) * len(translators)

path = os.path.join(destination_folder, 'diagrams')
folder = os.path.join(content_root, path)
os.makedirs(folder, exist_ok=True)
logger.info(f"Rendering diagrams to {os.path.abspath(folder)}")

with tqdm(total=diagram_count, dynamic_ncols=True, unit='Diagram') as pbar:
    for lang, translator in translators.items():
        for key, diagram in data['diagrams'].items():
            filename = f"{key}-{sanitize_filename(diagram['name'])}-{lang}.drawio"
            file = os.path.join(folder, filename)

            pbar.set_description(f"Generating diagram {key} '{diagram['name']}' [{lang}] to {file}")
            draw_io_xml = drawiodiagram.create_diagram(key, jsmodel, translator)
            with open(file, 'wb') as out:
                out.write(etree.tostring(draw_io_xml))
                logger.debug(f"Diagram '{diagram['name']}' stored in draw.io format to {filename}")
            pbar.update(1)

In [ ]:
if config_folder_before is not None:
    logger.warning(f"Moving configuration folder back to {config_folder_before}")
    os.rename(odm_config_folder, config_folder_before)

In [ ]:
logger.info(f"Successfully created {diagram_count} diagrams to {os.path.join(destination_folder, 'diagrams')}")

# Create web content

In [ ]:
from IM_WEB import listWebdoku
from IM_WEB.IM_HTML.printHTML import HTMLExport
from SSOT_db.IM_OBJECTS import Languagetext

html_export = HTMLExport()
html_export.setmodel(jsmodel)

repo_root = os.getcwd()

if not os.path.isdir(os.path.join(repo_root, 'pythonWork')):
    repo_root = os.path.join(repo_root, '..', '..')

repo_root = os.path.abspath(os.path.join(repo_root, 'pythonWork', 'pythonSource', 'IM_WEB', 'html-lib'))
html_export.setWebDirec(destination_folder)
logger.info(f"Web ressources {html_export.libSourceDirec}")
listWebdoku.listwebmain(html_export)

In [ ]:
## Custom link injection routine
import re

udpr_pattern = arguments.udpr_pattern

if udpr_pattern is not None:
    logger.info(f"Using pattern {udpr_pattern} to inject hyperlinks")
    
    def add_link_by_pattern(element: dict) -> (str or None):
        udpr = element.get('userdefprops')
        if udpr is not None:
            assert isinstance(udpr, dict) 
            for m_key, m_value in udpr.items():
                for g_key, g_value in m_value.items():
                    for key, value in g_value.items():
                        print(f"UDPR {m_key} {g_key}: {key} == {value}") 
                        if pattern.match(value['name']):
                            return value['value']
        return None

    pattern = re.compile(udpr_pattern)
    html_export.custom_hyperlink_extractor = add_link_by_pattern

In [ ]:
from IM_WEB.IM_HTML.svgpublisher import publish_svg_diagrams

for lang in languages:
    logger.info(f"Producing diagrams for {lang}")
    publish_svg_diagrams(html_export, lang)

# Sparx Enterprise Architect Export
Creates a XMI (XMI 2.1 / UML 2.1) export 
suitable to import the model into Sparx Enterprise Architect. 

In [ ]:
from IM_EA.export.xmiexport import XMIBuilder
from lxml import etree

if arguments.sparx_ea or arguments.all:
    exporter = XMIBuilder(jsmodel, languages[0])
    tree = exporter.model_to_basic_xmi()
    exporter.model_to_ea_extension()

    destination = os.path.join(destination_folder, f"{data['model']['name']}.xmi")

    et = etree.ElementTree(tree)
    et.write(destination, pretty_print=True)
    logger.info(f"Exported for Sparx Enterprise Architect to {os.path.abspath(destination)}. Language '{languages[0]}'")

# Report complete

In [ ]:
print(f"\x1b[32mSucessfully\x1b[39m produced documentation in {os.path.abspath(destination_folder)}")